In [ ]:
# Supplementary Figure 2C-D: tracheolar volume fractions across neuropil super-regions
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

fraction_file = Path('fig2_outputs/tracheation_by_neuropil.csv')
out_dir = Path('supp_outputs')
out_dir.mkdir(parents=True, exist_ok=True)

superregions = {
    'OL': ['ME_L', 'ME_R', 'AME_L', 'AME_R', 'LO_L', 'LO_R', 'LOP_L', 'LOP_R'],
    'MB': ['MB_CA_R', 'MB_CA_L', 'MB_PED_L', 'MB_PED_R', 'MB_VL_R', 'MB_VL_L', 'MB_ML_R', 'MB_ML_L'],
    'CX': ['FB', 'EB', 'PB', 'NO'],
    'LX': ['BU_L', 'BU_R', 'LAL_R', 'LAL_L', 'GA_R', 'GA_L'],
    'VLNP': ['AOTU_R', 'AOTU_L', 'PVLP_L', 'AVLP_L', 'PVLP_R', 'AVLP_R', 'PLP_L', 'PLP_R', 'WED_L', 'WED_R'],
    'LH': ['LH_L', 'LH_R'],
    'SNP': ['SLP_L', 'SLP_R', 'SIP_L', 'SIP_R', 'SMP_R', 'SMP_L'],
    'INP': ['CRE_L', 'CRE_R', 'ICL_L', 'SCL_L', 'ICL_R', 'SCL_R', 'IB_L', 'IB_R', 'ATL_R', 'ATL_L'],
    'AL': ['AL_L', 'AL_R'],
    'VMNP': ['VES_L', 'VES_R', 'EPA_R', 'EPA_L', 'GOR_L', 'GOR_R', 'SPS_L', 'SPS_R', 'IPS_R', 'IPS_L'],
    'PENP': ['SAD', 'AMMC_R', 'AMMC_L', 'FLA_R', 'FLA_L', 'CAN_R', 'CAN_L', 'PRW'],
    'GNG': ['GNG'],
    'OCG': ['OCG'],
}

region_data = pd.read_csv(fraction_file).set_index('neuropil')
rows = []
for region, neuropils in superregions.items():
    group = region_data.loc[region_data.index.intersection(neuropils)]
    trachea_volume = group['trachea_volume_um3'].sum()
    neuropil_volume = group['neuropil_volume_um3'].sum()
    rows.append({'super_region': region, 'trachea_volume_um3': trachea_volume,
                 'neuropil_volume_um3': neuropil_volume,
                 'fraction_percent': 100 * trachea_volume / neuropil_volume})
summary = pd.DataFrame(rows).sort_values('fraction_percent', ascending=False)

plt.rcParams.update({'font.family': 'Arial', 'font.size': 9, 'svg.fonttype': 'none'})
fig, ax = plt.subplots(figsize=(5.2, 4.2))
colors = plt.cm.GnBu(np.linspace(.3, .95, len(summary)))
ax.barh(summary['super_region'], summary['fraction_percent'], color=colors)
ax.invert_yaxis()
ax.set_xlabel('Tracheole volume fraction (%)')
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig(out_dir / 'supp_fig2c.svg', bbox_inches='tight')
plt.show()

mesh_dir = Path('data/neuropil_mesh_cache')
fractions = summary.set_index('super_region')['fraction_percent'].to_dict()
region_to_super = {name: region for region, names in superregions.items() for name in names}
mesh_data = {}
for name in region_data.index:
    with np.load(mesh_dir / f'{name}.npz') as mesh:
        mesh_data[name] = (np.asarray(mesh['vertices'], float), np.asarray(mesh['faces'], int))

norm = Normalize(vmin=0, vmax=.15)
cmap = plt.get_cmap('GnBu')
all_vertices = np.vstack([v for v, _ in mesh_data.values()])
low, high = all_vertices.min(axis=0), all_vertices.max(axis=0)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for ax, (title, horizontal, vertical, depth, flip) in zip(axes,
        [('Frontal', 0, 1, 2, False), ('Dorsal', 0, 2, 1, True)]):
    triangles, depths, colors = [], [], []
    for name, (vertices, faces) in mesh_data.items():
        faces = faces[np.linspace(0, len(faces)-1, min(6000, len(faces)), dtype=int)]
        tri = vertices[faces]
        triangles.append(tri[:, :, [horizontal, vertical]])
        depths.append(tri[:, :, depth].mean(axis=1))
        colors.append(np.tile(cmap(norm(fractions[region_to_super[name]])), (len(tri), 1)))
    triangles, depths, colors = np.concatenate(triangles), np.concatenate(depths), np.concatenate(colors)
    order = np.argsort(-depths)
    ax.add_collection(PolyCollection(triangles[order], facecolors=colors[order],
                                     edgecolors='none', alpha=.96, rasterized=True))
    dx, dy = high[horizontal] - low[horizontal], high[vertical] - low[vertical]
    ax.set_xlim(low[horizontal]-.025*dx, high[horizontal]+.025*dx)
    ax.set_ylim(low[vertical]-.025*dy, high[vertical]+.025*dy)
    if flip:
        ax.invert_yaxis()
    ax.set_aspect('equal')
    ax.set_axis_off()
    ax.set_title(title)
fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=axes, fraction=.025,
             label='Tracheole volume fraction (%)')
fig.savefig(out_dir / 'supp_fig2d.svg', bbox_inches='tight')
plt.show()
